# 03. 방법 1: 제로샷 분류

학습 없이 이미지 임베딩과 국가별 문장 임베딩의 코사인 유사도로 분류한다. 런타임: GPU.

1. 국가별 문장 생성: `a street photo taken in {country}`
2. 이미지 정규화, 문장 토큰화
3. 이미지 임베딩과 20개 문장 임베딩의 코사인 유사도에 CLIP의 온도(logit scale)를 곱해 softmax
4. 확률이 가장 높은 국가를 예측으로 하고 상위 5개를 함께 기록

In [ ]:
# 공통 셋업: 저장소 clone, 패키지 설치, Drive 연결
import os, sys
REPO_URL = "https://github.com/Cranezero/aix-assignment-pr.git"
REPO_DIR = "/content/aix-assignment-pr"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -r requirements.txt
sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader

from src import config
from src.clip_utils import PROMPT_TEMPLATES, build_prompts, encode_images, encode_texts, load_clip, zeroshot_probs
from src.data import StreetDataset, get_transform, load_split, prepare_images
from src.metrics import evaluate, plot_confusion, save_results, top_confusions
from src.utils import export_outputs, get_device, restore_outputs, set_seed

restore_outputs()  # results/, assets/ 폴더 생성 + Drive에 저장된 이전 결과 가져오기

set_seed(config.SEED)
device = get_device()
image_dir = prepare_images()
model, processor = load_clip(device)
logit_scale = model.logit_scale.exp().item()
print(device, "logit scale:", round(logit_scale, 2))

In [ ]:
test = load_split("test")
loader = DataLoader(StreetDataset(test, image_dir, get_transform(train=False)),
                    batch_size=256, shuffle=False, num_workers=2)
t0 = time.time()
img_emb, labels = encode_images(model, loader, device)
encode_sec = time.time() - t0
print(img_emb.shape, f"{encode_sec:.1f}s")

## 프롬프트 비교

같은 이미지 임베딩에 세 가지 문장 템플릿을 적용한다.

In [ ]:
rows, probs_by_template = [], {}
for key, template in PROMPT_TEMPLATES.items():
    text_emb = encode_texts(model, processor, build_prompts(key), device).cpu()
    probs = zeroshot_probs(img_emb, text_emb, logit_scale).numpy()
    probs_by_template[key] = probs
    m = evaluate(probs, labels.numpy())
    rows.append({"template": template.format("{country}"), "example": template.format("South Korea"),
                 "top1": m["top1"], "top5": m["top5"], "macro": m["macro"]})
prompt_table = pd.DataFrame(rows)
prompt_table.to_csv(config.RESULTS_DIR / "zeroshot_prompts.csv", index=False)
display(prompt_table.round(4))

## 기본 프롬프트 결과

이후 비교에는 `a street photo taken in {country}`를 쓴다.

In [ ]:
probs = probs_by_template["street"]
metrics = save_results("zeroshot", test["id"], probs, labels.numpy(),
                       extra={"prompt": PROMPT_TEMPLATES["street"], "train_time_sec": 0.0,
                              "test_encode_sec": encode_sec})
print({k: round(v, 4) for k, v in metrics.items() if k in ("top1", "top5", "macro")})
per_country = pd.Series(metrics["per_country"]).rename("acc").to_frame()
per_country.insert(0, "name", [config.COUNTRIES[c] for c in per_country.index])
display(per_country.sort_values("acc", ascending=False).round(3))

In [ ]:
y_true = [config.CODES[i] for i in labels.numpy()]
y_pred = [config.CODES[i] for i in probs.argmax(1)]
fig, ax = plt.subplots(figsize=(9, 8))
plot_confusion(y_true, y_pred, ax=ax, title="Zero-shot (row-normalized)")
plt.tight_layout()
plt.savefig(config.ASSETS_DIR / "confusion_zeroshot.png", dpi=120, bbox_inches="tight")
plt.show()
display(top_confusions(y_true, y_pred, n=10).round(3))

## 예측 예시

사진 한 장과 상위 5개 예측 확률.

In [ ]:
def show_prediction(i, path):
    row = test.iloc[i]
    top = np.argsort(-probs[i])[:5]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1.6, 1]})
    a1.imshow(Image.open(image_dir / f"{row['id']}.jpg")); a1.axis("off")
    a1.set_title(f"True: {config.COUNTRIES[row['country']]}")
    a2.barh([config.COUNTRIES[config.CODES[j]] for j in top][::-1], probs[i, top][::-1])
    a2.set_xlim(0, 1); a2.set_xlabel("probability")
    plt.tight_layout(); plt.savefig(path, dpi=110, bbox_inches="tight"); plt.show()

rng = np.random.default_rng(config.SEED)
correct = np.where(probs.argmax(1) == labels.numpy())[0]
show_prediction(rng.choice(correct), config.ASSETS_DIR / "zeroshot_example.png")

In [ ]:
export_outputs()